### N-Gram Analysis (Yoruba Text)

N-gram analysis is a basic yet powerful technique in Natural Language Processing (NLP). It involves breaking text into sequences of `n` items, usually words, to understand patterns, frequency, and co-occurrence. Examples include:
- **Unigram** (n=1): single words
- **Bigram** (n=2): pairs of words
- **Trigram** (n=3): triples of words
- ...and so on.

In this notebook, we’ll apply n-gram analysis to Yoruba-language text. We use the Yankari dataset (https://huggingface.co/datasets/acflp/YANKARI) as the reference corpus, extract n-grams of varying sizes, compute their probabilities, and use them to score new texts based on how likely they are under the model.

In [1]:
# import libraries
import json
import math
import regex as re
import unicodedata
from collections import Counter
from datasets import load_dataset
import pandas as pd
import os
import pickle

In [2]:
# Load the reference corpus into a pandas DataFrame
dataset = load_dataset("acflp/YANKARI", verification_mode="basic_checks", trust_remote_code=True)

reference_text = pd.DataFrame(dataset["train"])[['text']]

# Display the first 5 rows of the DataFrame
print(reference_text.head())

                                                text
0  O ma ṣe o! Ijamba ọkọ ofurufu gba ẹmi eeyan ma...
1  Ọbasanjọ ni ki ijọba Naijiria lọ gba amọran lo...
2  Awọn ajinigbe ha ni Kwara, ọkan ku: Agbarijọ a...
3  AbdulRazaq san gbogbo gbese ti Gomina Ahmẹd jẹ...
4  Ọmọ ile igbimọ aṣoju-ṣofin tẹlẹ ni Kwara fẹgbẹ...


#### Preprocessing and Tokenization
To build meaningful n-grams, we first normalize the text and tokenize it.
We:
- Normalize characters using Unicode NFC
- Lowercase the text
- Remove non-letter characters except hyphens and apostrophes
- Tokenize by matching word-like sequences

In [3]:
def preprocess(text):
    if not isinstance(text, str):
        return ""
    text = unicodedata.normalize('NFC', text)
    text = text.lower()
    text = re.sub(r"[^\p{L}\s'-]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def tokenize_text(text):
    if isinstance(text, pd.Series) or isinstance(text, list):
        tokens = []
        for row in text:
            row = preprocess(row)
            tokens.extend(re.findall(r"\p{L}+(?:['-]\p{L}+)*", row))
        return tokens
    elif isinstance(text, str):
        text = preprocess(text)
        return re.findall(r"\p{L}+(?:['-]\p{L}+)*", text)
    else:
        return []

In [4]:
# Use the function to get all tokens from the reference text
all_tokens = tokenize_text(reference_text['text'])
print(f"Total tokens collected: {len(all_tokens)}")

Total tokens collected: 13586924


##### Tokenization Output
In the code block above, we create a variable to store all tokens from the reference corpus, apply preprocessing and tokenization, and print the total number of word tokens extracted.

We collected over **13 million tokens** after cleaning and tokenizing the text. These tokens form the basis for generating n-gram models (e.g., bigrams, trigrams) in later steps.


#### Building and Saving N-grams

To generate n-grams of different lengths (bigrams, trigrams, and 4-grams), we use the `build_multiple_ngrams()` function. It accepts the token list and a list of `n` values, and returns a dictionary mapping each `n` to its corresponding n-gram list.

We then save each n-gram list to a separate `.jsonl` file using `save_ngrams_to_jsonl()` function.

In [5]:
def build_multiple_ngrams(tokens, n_values=[2, 3, 4]):
    """
    Build n-grams for multiple n values from a list of tokens.
    Returns a dict: {n: [list of n-grams]}
    """
    ngram_dict = {}
    for n in n_values:
        ngram_dict[n] = [tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1)]
    return ngram_dict

In [6]:
def save_ngrams_to_jsonl(ngram_dict, folder="data"):
    """
    Save each n-gram list in ngram_dict to a separate JSONL file.
    Args:
        ngram_dict: Dictionary where keys are n (2, 3, 4) and values are lists of n-grams (tuples).
        folder: Folder to save the files in (default: "data").
    """
    import os
    import json
    os.makedirs(folder, exist_ok=True)
    for n, ngrams in ngram_dict.items():
        filename = f"{folder}/ngrams_{n}gram.jsonl"
        with open(filename, "w", encoding="utf-8") as f:
            for ngram in ngrams:
                json.dump(ngram, f, ensure_ascii=False)
                f.write("\n")
        print(f"Saved {len(ngrams)} {n}-grams to {filename}")
    print("Done saving n-gram files.")

In [7]:
# Build all n-grams at once and save them
ngram_dict = build_multiple_ngrams(all_tokens, n_values=[2, 3, 4])
save_ngrams_to_jsonl(ngram_dict)

Saved 13586923 2-grams to data/ngrams_2gram.jsonl
Saved 13586922 3-grams to data/ngrams_3gram.jsonl
Saved 13586921 4-grams to data/ngrams_4gram.jsonl
Done saving n-gram files.


#### Computing N-gram Probabilities
We load the previously saved `.jsonl` n-gram files, count their frequencies using Python’s `Counter`, and compute their probabilities.

The probability of each n-gram is calculated as:
$$
P(w_1, w_2, ..., w_n) = \frac{\text{count}(w_1, ..., w_n)}{\text{total number of n-grams}}
$$

We repeat this process for bigrams (n=2), trigrams (n=3), and 4-grams (n=4).

After computing probabilities, we save each dictionary using Python’s `pickle` module. This allows for faster loading during evaluation, especially for large files.


In [7]:
# Compute n-gram probability dicts for 2, 3, and 4-grams before saving with pickle
from collections import Counter

def load_ngrams_jsonl(filename):
    ngrams = []
    with open(filename, "r", encoding="utf-8") as f:
        for line in f:
            ngram = tuple(json.loads(line))
            ngrams.append(ngram)
    return ngrams

# 2-gram
grams_2 = load_ngrams_jsonl("C:/Users/DELL/Desktop/translation_evaluator/data/ngrams_2gram.jsonl")
counter_2 = Counter(grams_2)
total_2 = sum(counter_2.values())
ngram_probs_2 = {ngram: count / total_2 for ngram, count in counter_2.items()}

# 3-gram
grams_3 = load_ngrams_jsonl("C:/Users/DELL/Desktop/translation_evaluator/data/ngrams_3gram.jsonl")
counter_3 = Counter(grams_3)
total_3 = sum(counter_3.values())
ngram_probs_3 = {ngram: count / total_3 for ngram, count in counter_3.items()}

# 4-gram
grams_4 = load_ngrams_jsonl("C:/Users/DELL/Desktop/translation_evaluator/data/ngrams_4gram.jsonl")
counter_4 = Counter(grams_4)
total_4 = sum(counter_4.values())
ngram_probs_4 = {ngram: count / total_4 for ngram, count in counter_4.items()}

In [8]:
with open("C:/Users/DELL/Desktop/translation_evaluator/data/ngrams_2gram_probs.pkl", "wb") as f:
    pickle.dump(ngram_probs_2, f, protocol=pickle.HIGHEST_PROTOCOL)

with open("C:/Users/DELL/Desktop/translation_evaluator/data/ngrams_3gram_probs.pkl", "wb") as f:
    pickle.dump(ngram_probs_3, f, protocol=pickle.HIGHEST_PROTOCOL)

with open("C:/Users/DELL/Desktop/translation_evaluator/data/ngrams_4gram_probs.pkl", "wb") as f:
    pickle.dump(ngram_probs_4, f, protocol=pickle.HIGHEST_PROTOCOL)

#### Scoring New Texts Using Log-Probability
Now, to evaluate how closely a new text matches the reference corpus & domain, we use n-gram log-probability scoring.

- Why Log Probabilities? When computing the probability of a sequence of n-grams, multiplying many small probabilities results in extremely tiny numbers. This leads to underflow issues.
Instead of multiplying, we use the log of each probability and sum them:
$$
\log P_{\text{total}} = \sum_{i=1}^{k} \log P(w_i, w_{i+1}, ..., w_{i+n-1})
$$

In [9]:
def compute_log_prob(n_grams, n_gram_probs, smoothing=1e-8):
    log_prob = 0.0
    for n_gram in n_grams:
        ngram_key = str(n_gram)
        prob = n_gram_probs.get(ngram_key, smoothing)
        log_prob += math.log(prob)
    return log_prob

In [10]:
def get_score_for_new_text(new_text, ngram_probs_dict, n_value, smoothing=1e-8):
    tokens = tokenize_text(preprocess(new_text))
    ngrams = []
    for i in range(len(tokens) - n_value + 1):
        ngram = tuple(tokens[i:i + n_value])
        ngrams.append(ngram)
    log_prob_score = compute_log_prob(ngrams, ngram_probs_dict, smoothing=smoothing)

    return log_prob_score


#### How Does The Scoring Work
1. The new text is preprocessed and tokenized.
2. Its n-grams are generated **in memory** — we don’t save them to disk because we’re not hoarders.
3. For each n-gram, we **look up its probability** in the reference model.
4. If an n-gram doesn’t exist, we use a small smoothing value.
5. The log-probabilities are **summed** to give a final score.

### Interpreting the Score
- The log-probability is typically a **large negative number**.
- The **less negative** the score (i.e., closer to zero), the **more similar** the text is to the reference corpus.

##### Let's test this out
- Text A is a sample from Wura dataset: https://huggingface.co/datasets/castorini/wura
- Text B is a sample taken from a Yorùbá legal text. Doc Name: Iwe Ofin Awọn ti wọn fi Ipabalopọ 


In [11]:
# Text from Wura dataset: 
text_a = "Iroyin faakaja kan n lọ lọwọ lori ẹrọ ayelujara bayii, awọn ọjọgbọn to mọ nipa orin atawọn ọjẹwẹwẹ ti wọn maa n gbadun orin ni wọn n fa ọrọ naa mọra wọn lọwọ lori ẹrọ ayelujara.Ohun ti wọn n fa mọ ara wọn lọwọ naa ni pe tani agba ninu awọn olori yii, Mama Bọla Arẹ ati Tọpẹ Alabi. Onkọrin ẹsin Kristẹni lawọn mejeeji, bo tiẹ jẹ pe ọkan ju ekeji lọ lọjọ ori, ṣugbọn ipa tawọn mejeeji ko ninu orin kiko ko ṣee ko kere rara.Ṣugbọn ohun tawọn eeyan n bi ara wọn leere ni pe tani o mọ orin i kọ ju ninu awọn mejeeji.Bawọn kan ti fowo mu Tọpẹ Alabi, bẹẹ lawọn mi in fowo mu Bọla Arẹ.Ọpọ awọn agba ti wọn ti wa tipẹ lo fọwọ mu Bọla Arẹ, bẹẹ lọpọ awọn ọdọ ode oni fọwọ mu Tọpẹ Alabi.Gbajumọ onkọwe kan, Mọlara Woods tilẹ sọ pe lọdun 2004, Mama Bọla Arẹ ati ẹgbẹ obinrin rere, iyẹn Good Women Choir, ti Mama Faṣọyin jẹ adari rẹ ṣere kan niluu London, fọnfọn ni gbọngan Queen Elizabeth naa kun lọjọ naa lọhun un.O daa, ta lẹyin mu ninu awọn mejeeji yii, ṣe Bọla Arẹ ni abi Tọpẹ Alabi, ta le maa pe sode ere yin?"

In [12]:
text_b = "Nigba ti wọn ba n ṣe ayẹwo tabi ifọrọwanilẹnuwo pẹlu agbofinro, awọn abanirojọ, tabi  awọn agbẹjọro olugbeja, o ni ẹtọ lati kan si alagbawi pẹlu onigbeja ibalopọ. Bi o ti lẹjẹ  pe alatilẹhin awọn ti lu jamba ko le funni imọran ofin, wọn le funni ni atilẹyin,  iranlọwọ pẹlu eto aabo, so awọn olufaragba pọ pẹlu awọn orisun afikun ati/tabi awọn  itọkasi, ṣe iranlọwọ pẹlu eto aabo fun awọn ti o lu jamba ni oye awọn aṣayan wọn, ati  ṣalaye ohun ti wọn le reti ninu eto idajọ ọdaràn.  Awọn ibaraẹnisọrọ eyikeyi ti o ni pẹlu agbaroso fun eni lu jamba ifipabanilopọ yoo jẹ  aṣiri ati ohun anfani, jẹ wọn kii yoo pin apakan eyikeyi ti ibaraẹnisọrọ rẹ pẹlu  oluyẹwo iṣoogun kan, ọlọpa kan, abanirojọ, tabi agbẹjọro olugbeja. Agbẹjọro eni lu  jamba rẹ yoo ṣe ayẹwo pẹlu rẹ lori awọn idiwọn lori aṣiri awọn ibaraẹnisọrọ rẹ pẹlu  wọn.  O ni ẹtọ lati mọ nipa ẹtọ rẹ lati kan si alagbawi ẹni lu jamba nipasẹ oluyẹwo iṣoogun  kan, oṣiṣẹ agbofinro, tabi abanirojọ ipinlẹ ṣaaju ki o to beere ibeere eyikeyi nipa iriri  rẹ. Ko si iru iwadii tabi ifọrọwanilẹnuwo bẹẹ ti yoo maa ba a lọ ayafi bi ẹni naa ba  mọọmo fi ẹtọ yii silẹ. Wo Awọn ofin gbogbo § 23-98-5(a)(1).  Koda ti o ba pinnu lati ma ṣe alagbawo pẹlu alagbawi fun ẹni lu jamba ni igbakuba  ti iwadii waaye lori ifipalopọ naa, o ni ẹtọ lati kan si alagbawi eni lu jamba ni gba  yoowu tii iwadii naa ba n lọ lọwọ. Wo R.I. Awọn ofin gbogbo § 23-98-5(a)(1).  Awọn agbejọro wa lati ba ọ sọrọ nigbakugba nipasẹ Victims of Crime Helpline  alaṣiri ti 1-800-494-8100 ti o wa ni gbogbo ipinlẹ Rhode Island. Awọn afikun awọn  isofunni to le wulo fun ọ wa ni opin iwe yii."

In [13]:
with open("C:/Users/DELL/Desktop/translation_evaluator/data/ngrams_2gram_probs.pkl", "rb") as f:
    ngram_probs_2 = pickle.load(f)
with open("C:/Users/DELL/Desktop/translation_evaluator/data/ngrams_3gram_probs.pkl", "rb") as f:
    ngram_probs_3 = pickle.load(f)
with open("C:/Users/DELL/Desktop/translation_evaluator/data/ngrams_4gram_probs.pkl", "rb") as f:
    ngram_probs_4_fast = pickle.load(f)

In [14]:
#scoring for text_a
score_bigram_ta = get_score_for_new_text(text_a, ngram_probs_2, n_value=2)
score_trigram_ta = get_score_for_new_text(text_a, ngram_probs_3, n_value=3)
score_fourgram_ta = get_score_for_new_text(text_a, ngram_probs_4, n_value=4)

# print scores for text_a
print(f"2-gram score: {score_bigram_ta}")

print(f"3-gram score: {score_trigram_ta}")

print(f"4-gram score: {score_fourgram_ta}")

2-gram score: -3757.818871766272
3-gram score: -3739.3981910223197
4-gram score: -3720.9775102783674


In [15]:
#scoring for text_b
score_bigram_tb = get_score_for_new_text(text_b, ngram_probs_2, n_value=2)
score_trigram_tb = get_score_for_new_text(text_b, ngram_probs_3, n_value=3)
score_fourgram_tb = get_score_for_new_text(text_b, ngram_probs_4, n_value=4)

# print scores for text_b
print(f"2-gram score: {score_bigram_tb}")

print(f"3-gram score: {score_trigram_tb}")

print(f"4-gram score: {score_fourgram_tb}")

2-gram score: -5434.100819465931
3-gram score: -5415.680138721978
4-gram score: -5397.259457978026


| N-gram Size | Text A (Wura) | Text B (Legal) |
|-------------|---------------|----------------|
| 2-gram      | -3757.82      | -5434.10       |
| 3-gram      | -3739.40      | -5415.68       |
| 4-gram      | -3720.98      | -5397.26       |

This results suggests that Text A is more **linguistically similar** to the reference corpus.